In [13]:
#!/usr/bin/env python
# coding: utf-8

# Inference Notebook

This notebook runs the actual inference step for our final submission. After training our models in `training.ipynb`, this is a way for us to generate predictions on the `test.csv` dataset. 

Our overall architecture is a 3-level stacking ensemble. We save all of the trained models and preprocessing artifacts during training so that we can load them back in here. This makes sure the test data undergoes the exact same transformations as our training data did. We also avoid data leakage since our normalizations/imputations rely only on parameters learned from the training set.

### Usage
- Double check that `RUN_DIR` points to the model folder you want to evaluate.
- Make sure the Kaggle `test.csv` is in the same directory.
- Run all cells to produce the `submission.csv` file.

In [14]:
# Loads all Level 1 + Level 2 models and artifacts from a training run
# directory, applies the identical preprocessing as test3.py, runs the
# full 3-level stacking ensemble on test.csv, and outputs a 2-column
# CSV (INDEX_NR, INDICATED_DAMAGE) sorted by INDEX_NR ascending.

from pathlib import Path
import gc
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.experimental import enable_iterative_imputer
from sklearn.base import BaseEstimator, TransformerMixin


# custom transformer to safely drop columns inside an sklearn pipeline
class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, keep_indices):
        # store the specific column indices we want to keep
        self.keep_indices = keep_indices

    def fit(self, X, y=None):
        # required dummy method so scikit-learn pipelines dont crash
        return self

    def transform(self, X):
        if hasattr(X, "iloc"):
            return X.iloc[:, self.keep_indices]
        return X[:, self.keep_indices]


warnings.filterwarnings("ignore")


## Configuration
First, we point `RUN_DIR` to the folder containing our trained artifacts. The code will automatically look for `test.csv` in the root directory or in a nested folder, just to handle any differences in where we each put our files locally.

In [15]:
# CONFIGURATIONS

# Path to the training run directory containing models/ and metadata files.
RUN_DIR = Path("runs/2026-04-26_16-52-21")

# Base directory for data files (train.csv, test.csv).
BASE_DIR = Path(".")
if not (BASE_DIR / "train.csv").exists():
    BASE_DIR = Path("ml final project")

TARGET_COL = "INDICATED_DAMAGE"
MISSING_CAT_TOKEN = "__MISSING__"


## Loading Artifacts from Training
Before we do anything with the test data, we need to load all the rules we learned during training. We pull:
- The exact feature subsets each model expects.
- The categorical mappings to keep pandas happy.
- Optimized Level 3 blending weights, decision threshold that maximizes our F1 score.

In [16]:
# RUN ARTIFACTS

print(f"Loading run artifacts from: {RUN_DIR}")

with open(RUN_DIR / "run_metadata.json") as f:
    run_metadata = json.load(f)

with open(RUN_DIR / "feature_subsets.json") as f:
    feature_subsets_json = json.load(f)  # {"fs1": [...], "fs2": [...], ...}

with open(RUN_DIR / "models" / "l3_weights.json") as f:
    l3_weights = json.load(f)  # {"l2_logistic_baseline": 0.5, "l2_xgb_standard": 0.5}

with open(RUN_DIR / "models" / "l3_best_threshold.json") as f:
    l3_threshold = json.load(f)["best_threshold"]

LEVEL1_MODEL_NAMES = run_metadata["level1_model_names"]
LEVEL2_MODEL_NAMES = run_metadata["level2_model_names"]
N_FOLDS = run_metadata["n_folds"]
N_FEATURE_SUBSETS = run_metadata["n_feature_subsets"]

print(f"  Level 1 models: {len(LEVEL1_MODEL_NAMES)}")
print(f"  Level 2 models: {len(LEVEL2_MODEL_NAMES)}")
print(f"  Folds: {N_FOLDS}")
print(f"  Feature subsets: {N_FEATURE_SUBSETS}")
print(f"  L3 weights: {l3_weights}")
print(f"  L3 threshold: {l3_threshold:.6f}")


Loading run artifacts from: runs/2026-04-26_16-52-21
  Level 1 models: 30
  Level 2 models: 2
  Folds: 3
  Feature subsets: 2
  L3 weights: {'l2_logistic_baseline': 0.5, 'l2_xgb_standard': 0.5}
  L3 threshold: 0.323333


## Cleaning & Preprocessing (Test Set)
We have to apply the exact same cleaning steps to `test.csv` that we used on the training set, otherwise our models will get confused by unseen categories or unformatted text. 

Our main cleaning pipeline handles things like:
- **Time of Day Inference:** If `TIME_OF_DAY` is missing, we try to guess it from the `TIME` column.
- **Free-Text Indicators:** We turn columns with random free-text notes into simple boolean flags (1 if a note exists, 0 otherwise).
- **Missing Value Standardization:** We consolidate all empty strings, NaNs, and "Unknown" values into a unified missing token so our tree models treat them consistently.
- **Dropping Leakage:** We drop identifiers like report numbers that have no real predictive power.

In [17]:
# CLEANING FUNCTIONS (identical to test3.py)


# standardizes column names to uppercase and strips whitespace
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip().upper() for c in out.columns]
    return out


# buckets raw time values into categorical times of day
def infer_time_of_day(time_value):
    if pd.isna(time_value):
        return np.nan
    text = str(time_value).strip()
    if not text:
        return np.nan
    try:
        hour = int(text.split(":")[0])  # just grab the hour from hh:mm
    except Exception:
        return np.nan

    if 5 <= hour <= 6:
        return "Dawn"
    if 7 <= hour <= 17:
        return "Day"
    if 18 <= hour <= 19:
        return "Dusk"
    return "Night"


# checks if a text field actually has content in it
def has_text_flag(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.strip().ne("").astype(int)


# forces all the weird null representations into a single missing token
def normalize_cat_series(series: pd.Series, missing_token: str = MISSING_CAT_TOKEN) -> pd.Series:
    s = series.astype("string").str.strip()
    s = s.fillna(missing_token)
    s = s.replace({"": missing_token, "<NA>": missing_token, "nan": missing_token, "None": missing_token})
    return s


# the main cleaning function that applies all of our rules
def clean_wildlife_strike_df(df: pd.DataFrame, target_col: str = TARGET_COL) -> pd.DataFrame:
    d = normalize_columns(df)

    # Fill TIME_OF_DAY from TIME when missing, then drop TIME.
    if "TIME_OF_DAY" in d.columns and "TIME" in d.columns:
        tod_missing = d["TIME_OF_DAY"].isna() | d["TIME_OF_DAY"].astype(str).str.strip().eq("")
        d.loc[tod_missing, "TIME_OF_DAY"] = d.loc[tod_missing, "TIME"].apply(infer_time_of_day) # fill missing times using our helper
    elif "TIME" in d.columns:
        d["TIME_OF_DAY"] = d["TIME"].apply(infer_time_of_day) # fallback if time_of_day is completely missing

    # Encode free-text fields as missing-vs-not-missing indicators.
    if "LOCATION" in d.columns:
        d["LOCATION_HAS_TEXT"] = has_text_flag(d["LOCATION"])
    if "COMMENTS" in d.columns:
        d["COMMENTS_HAS_TEXT"] = has_text_flag(d["COMMENTS"])
    if "REMARKS" in d.columns:
        d["REMARKS_HAS_TEXT"] = has_text_flag(d["REMARKS"])

    # AC_MASS is ordinal (1..5).
    if "AC_MASS" in d.columns:
        d["AC_MASS"] = pd.to_numeric(d["AC_MASS"], errors="coerce")

    # Force requested categorical fields into categorical dtype.
    for col in [
        "ENG_1_POS",
        "ENG_2_POS",
        "ENG_3_POS",
        "ENG_4_POS",
        "REMAINS_COLLECTED",
        "REMAINS_SENT",
        "TIME_OF_DAY",
        "INCIDENT_MONTH",
    ]:
        if col in d.columns:
            d[col] = d[col].astype("string").astype("category")

    # Drop fields requested in notes.
    drop_cols = [
        "AIRPORT_ID",
        "LATITUDE",
        "LONGITUDE",
        "OPID",
        "REG",
        "FLT",
        # "AMA",      # kept for target encoding
        # "AMO",      # kept for target encoding
        # "EMA",      # kept for target encoding
        # "EMO",      # kept for target encoding
        #"SPECIES",
        # "SPECIES_ID",  # kept for target encoding
        "ENROUTE_STATE",
        "INDEX_NR",
        "TRANSFER",
        "TIME",
        "LOCATION",
        "COMMENTS",
        "REMARKS",
    ]
    d = d.drop(columns=[c for c in drop_cols if c in d.columns], errors="ignore") # clean out the junk columns we decided to drop

    if target_col in d.columns:
        target = pd.to_numeric(d[target_col], errors="coerce")
        if target.isna().any():
            mapped = (
                d[target_col]
                .astype(str)
                .str.strip()
                .str.upper()
                .map(
                    {
                        "1": 1,
                        "0": 0,
                        "YES": 1,
                        "NO": 0,
                        "Y": 1,
                        "N": 0,
                        "TRUE": 1,
                        "FALSE": 0,
                        "DAMAGE": 1,
                        "NO DAMAGE": 0,
                    }
                )
            )
            target = target.fillna(mapped)
        d[target_col] = target.astype("Int64")

    return d


## Loading Data
We now bring in `test.csv`. We make sure to save `INDEX_NR` off to the side because Kaggle needs it for the final submission format. We also keep a raw copy of the text columns since some of our Level 1 models are text-only TF-IDF pipelines that need the raw strings. Finally, we run the cleaning function from above.

In [18]:
# LOAD AND PREPROCESS TEST DATA

print(f"\n{'='*72}")
print("  Loading and preprocessing test data")
print(f"{'='*72}")

test_path = BASE_DIR / "test.csv"
test_df_raw = pd.read_csv(test_path)
print(f"Raw test data shape: {test_df_raw.shape}")

# Extract INDEX_NR before cleaning drops it
_temp = normalize_columns(test_df_raw)
test_index_nr = _temp["INDEX_NR"].copy()
print(f"INDEX_NR range: {test_index_nr.min()} — {test_index_nr.max()}")

# Extract raw REMARKS and COMMENTS text before cleaning drops it
X_text_test = (
    _temp["REMARKS"].fillna("").astype(str).str.strip()
    if "REMARKS" in _temp.columns
    else pd.Series("", index=test_df_raw.index)
)
X_text_comments_test = (
    _temp["COMMENTS"].fillna("").astype(str).str.strip()
    if "COMMENTS" in _temp.columns
    else pd.Series("", index=test_df_raw.index)
)
del _temp

# Apply cleaning
test_df = clean_wildlife_strike_df(test_df_raw)

# Drop target if present (shouldn't be in test.csv, but safety check)
if TARGET_COL in test_df.columns:
    test_df = test_df.drop(columns=[TARGET_COL])

X_test = test_df.copy()



  Loading and preprocessing test data
Raw test data shape: (34131, 54)
INDEX_NR range: 9000000 — 9034130


## Target Encoding & Type Identification
Some of our categorical variables have very high cardinality (lots of unique values). For these, we used Target Encoding during training to convert them into continuous numeric values. We load that trained encoder here and apply it to the test set. 

After that, we explicitly separate out our categorical and numeric columns. This is important because different models in our ensemble expect different input formats.

In [19]:
# TARGET ENCODING
te_path = RUN_DIR / "target_encoder.joblib"
TARGET_ENCODE_COLS = []
if te_path.exists():
    global_te = joblib.load(te_path)
    # Dynamically extract the features that were target encoded
    TARGET_ENCODE_COLS = list(global_te.feature_names_in_)

    print(f"Applying Target Encoding to {len(TARGET_ENCODE_COLS)} columns...")
    test_nan_mask = X_test[TARGET_ENCODE_COLS].isna()
    # apply target encoder to test set (casting to string to be safe)
    X_test[TARGET_ENCODE_COLS] = global_te.transform(X_test[TARGET_ENCODE_COLS].astype(str))
    X_test[TARGET_ENCODE_COLS] = X_test[TARGET_ENCODE_COLS].mask(test_nan_mask) # make sure nans stay nans
    for col in TARGET_ENCODE_COLS:
        X_test[col] = X_test[col].astype("float64")

# Identify column types (encoded columns will now be correctly typed as numeric)
categorical_cols = X_test.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_cols = [c for c in X_test.columns if c not in categorical_cols]

print(f"Cleaned test shape: {X_test.shape}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")


Applying Target Encoding to 7 columns...
Cleaned test shape: (34131, 44)
Categorical columns (27): ['INCIDENT_DATE', 'INCIDENT_MONTH', 'TIME_OF_DAY', 'RUNWAY', 'STATE', 'FAAREGION', 'OPERATOR', 'AC_CLASS', 'TYPE_ENG', 'ENG_1_POS', 'ENG_2_POS', 'ENG_3_POS', 'ENG_4_POS', 'PHASE_OF_FLIGHT', 'SKY', 'PRECIPITATION', 'BIRD_BAND_NUMBER', 'SPECIES', 'REMAINS_COLLECTED', 'REMAINS_SENT', 'WARNED', 'NUM_SEEN', 'NUM_STRUCK', 'SIZE', 'SOURCE', 'PERSON', 'LUPDATE']
Numeric columns (17): ['INCIDENT_YEAR', 'AIRPORT', 'AIRCRAFT', 'AMA', 'AMO', 'EMA', 'EMO', 'AC_MASS', 'NUM_ENGS', 'HEIGHT', 'SPEED', 'DISTANCE', 'SPECIES_ID', 'OUT_OF_RANGE_SPECIES', 'LOCATION_HAS_TEXT', 'COMMENTS_HAS_TEXT', 'REMARKS_HAS_TEXT']


## Categorical Normalization
Tree-based models like XGBoost, LightGBM, and CatBoost have native support for categorical features, but they are very strict: the test set categories MUST match the training set categories exactly. 

We load our saved `category_mappings.json` to force the test data to have the same category levels. If a new category shows up in the test set that we didn't see in training, it gets gracefully ignored or set to missing rather than crashing the pipeline.

In [20]:
# CATEGORICAL NORMALIZATION (aligned to test3.py mappings)

# XGBoost with enable_categorical=True strictly ensures that the categories order 
# from the pandas type matches the booster signature exactly.

print("Loading category metadata...")
cat_map_path = RUN_DIR / "category_mappings.json"
if not cat_map_path.exists():
    raise FileNotFoundError(f"Missing {cat_map_path}. Cannot align test categories without metadata.")

with open(cat_map_path) as f:
    category_mappings = json.load(f)

# Create native categorical views matching the strict sequence
X_test_native = X_test.copy()

# Apply mapping to test sets
for col in categorical_cols:
    X_test[col] = normalize_cat_series(X_test[col])
    X_test_native[col] = normalize_cat_series(X_test_native[col])

    if col in category_mappings:
        cats = category_mappings[col]
        # Force strict array match
        cat_type = pd.CategoricalDtype(categories=cats, ordered=False)
        X_test[col] = X_test[col].astype(cat_type)
        X_test_native[col] = X_test_native[col].astype(cat_type)

print("Categorical normalization complete (unified categories from training metadata).")


Loading category metadata...
Categorical normalization complete (unified categories from training metadata).


## Building Feature Subsets
To make our ensemble as diverse as possible, not all models look at the same data. We trained different models on different subsets of features. Here we generate those specific subsets for the test data based on the `feature_subsets.json` mapping. If a required column somehow got dropped, we fill it with NaNs so the model still has the right shape.

In [21]:
# BUILD FEATURE SUBSETS

print(f"\n{'='*72}")
print("  Building feature subsets")
print(f"{'='*72}")

# Ensure all columns from all unique subsets exist in test data
all_unique_cols = set()
for fs_cols in feature_subsets_json.values():
    all_unique_cols.update(fs_cols)

for col in all_unique_cols:
    if col not in X_test.columns:
        # if a feature is completely missing from the test set, pad it with nans
        X_test[col] = np.nan
        X_test_native[col] = np.nan



  Building feature subsets


## Imputing Missing Values
While our tree-based models can handle missing values naturally, some models in our pipeline (e.g. certain sklearn models) crash if they see a NaN. During training, we used an imputer to fill these gaps. We load the exact imputer here and fill in missing numeric values in the test set using the medians/means we learned from the training data.

In [22]:
# PRE-IMPUTE NUMERIC COLUMNS

# During training, all pipeline-model data (fs["X"]) was pre-imputed using
# the math (XGBRegressor) imputer.  Both pipeline_math and pipeline_tree_ohe
# models consumed this same pre-imputed data.  Native models used X_native
# which was NOT pre-imputed (they handle NaN natively).

# We replicate this here: load the math imputer, transform pipeline data.

print(f"\n{'='*72}")
print("  Pre-imputing numeric columns")
print(f"{'='*72}")

X_test_pipeline = X_test.copy()
global_imputer_path = RUN_DIR / "models" / "global_imputer.joblib"

if global_imputer_path.exists():
    print("  Applying global math imputer...")
    global_imputer = joblib.load(global_imputer_path)
    X_test_pipeline[numeric_cols] = global_imputer.transform(X_test_pipeline[numeric_cols].values)
    print(f"  Pre-imputation complete. Remaining NaN in pipeline numeric cols: 0")
else:
    print("  No global imputer found. Relying on per-fold pipeline imputers.")

gc.collect()



  Pre-imputing numeric columns
  No global imputer found. Relying on per-fold pipeline imputers.


714

## Prediction Helpers
Since we have a mix of model types (TF-IDF pipelines, native categorical models, and standard pre-imputed models), getting predictions out of them requires slightly different code for each. 

We make helper functions that route the data to the correct input format depending on the model's type:
- **Text models**: raw string series.
- **Native models**: get the dataset with strict categorical dtypes.
- **Pipeline models**: fully imputed numeric dataset.

In [23]:
# HELPER: PREDICT PROBABILITIES FROM A LOADED MODEL


# helper to extract probabilities regardless of model type
def predict_probs(model, X_eval):
    """Get class-1 probabilities from a fitted model."""
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_eval)[:, 1] # standard tree model probability
    else:
        scores = model.decision_function(X_eval)
        probs = 1.0 / (1.0 + np.exp(-scores)) # manually convert raw svm/linear scores to probabilities
    return np.clip(probs, 1e-7, 1.0 - 1e-7)


# figures out which preprocessing pipeline the model needs based on its name
def get_model_data_type(model_name: str) -> str:
    """Determine data type routing based on model name prefix."""
    if model_name == "text_logreg":
        return "text"
    if model_name == "text_comments_logreg":
        return "text_comments"
    if model_name.startswith("xgb_"):
        return "native"
    if model_name.startswith("lgbm_"):
        return "native"
    if model_name.startswith("catboost_"):
        return "native"
    if model_name.startswith("rf_"):
        return "pipeline"
    if model_name.startswith("lr_"):
        return "pipeline"
    if model_name.startswith("svm_"):
        return "pipeline"
    #
    print(f" unknown model type for '{model_name}', defaulting to pipeline")
    return "pipeline"


## Level 1 Inference (Base Models)
This is where the heavy lifting happens. For every single Level 1 model we trained, we load all of its cross-validation folds from disk. We run the test data through each fold and take the average probability. 

Averaging across folds makes our predictions more stable/robust to noise. The final matrix of all these Level 1 probabilities becomes the input feature set for our Level 2 models.

In [24]:
# LEVEL 1 INFERENCE

print(f"\n{'='*72}")
print("  Level 1 Inference")
print(f"{'='*72}")

n_test = len(X_test)
level2_input = pd.DataFrame(index=X_test.index)

for model_name in LEVEL1_MODEL_NAMES:
    data_type = get_model_data_type(model_name)

    # Determine input data
    if data_type == "text":
        X_input = X_text_test
    elif data_type == "text_comments":
        X_input = X_text_comments_test
    else:
        fs_cols = feature_subsets_json[model_name]

        if data_type == "native":
            X_input = X_test_native[fs_cols]
        else:  # pipeline
            X_input = X_test_pipeline

    # Load and average predictions across all folds
    fold_probs_sum = np.zeros(n_test, dtype=float)
    folds_loaded = 0

    for fold in range(1, N_FOLDS + 1):
        model_path = RUN_DIR / "models" / f"{model_name}_fold{fold}.joblib"
        if not model_path.exists():
            print(f"  ⚠ Missing: {model_path}")
            continue

        model = joblib.load(model_path)
        fold_probs = predict_probs(model, X_input)
        fold_probs_sum += fold_probs
        folds_loaded += 1

        del model
        gc.collect()

    if folds_loaded == 0:
        raise FileNotFoundError(f"No fold models found for {model_name}")

    avg_probs = fold_probs_sum / folds_loaded # take the average across all folds
    level2_input[model_name] = avg_probs # this becomes a feature for level 2

    print(
        f"  {model_name}: {folds_loaded} folds loaded, "
        f"mean_prob={avg_probs.mean():.4f}, "
        f"pred_pos_rate={(avg_probs >= 0.5).mean():.4f}"
    )

print(f"\nLevel 2 input matrix shape: {level2_input.shape}")

gc.collect()



  Level 1 Inference
  xgb_standard_fs1: 3 folds loaded, mean_prob=0.4834, pred_pos_rate=0.2068
  xgb_standard_fs2: 3 folds loaded, mean_prob=0.4536, pred_pos_rate=0.2061
  xgb_deep_slow_fs1: 3 folds loaded, mean_prob=0.4183, pred_pos_rate=0.0071
  xgb_deep_slow_fs2: 3 folds loaded, mean_prob=0.3610, pred_pos_rate=0.0035
  xgb_shallow_fast_fs1: 3 folds loaded, mean_prob=0.2819, pred_pos_rate=0.0249
  xgb_shallow_fast_fs2: 3 folds loaded, mean_prob=0.4770, pred_pos_rate=0.0044
  lgbm_standard_fs1: 3 folds loaded, mean_prob=0.1702, pred_pos_rate=0.0571
  lgbm_standard_fs2: 3 folds loaded, mean_prob=0.1786, pred_pos_rate=0.0519
  lgbm_feature_subsampler_fs1: 3 folds loaded, mean_prob=0.2117, pred_pos_rate=0.0881
  lgbm_feature_subsampler_fs2: 3 folds loaded, mean_prob=0.1374, pred_pos_rate=0.0000
  lgbm_row_subsampler_fs1: 3 folds loaded, mean_prob=0.1688, pred_pos_rate=0.0533
  lgbm_row_subsampler_fs2: 3 folds loaded, mean_prob=0.1588, pred_pos_rate=0.0001
  rf_deep_gini_fs1: 3 folds loa

0

## Injecting Raw Features into Level 2
Usually, Level 2 models only look at the probabilities outputted by Level 1. However, we found through experimentation that performance improved if we gave Level 2 some extra context: a few key categorical features like `AIRCRAFT`, `SIZE`, and `PERSON`. 

We apply the target encoder to these columns and append them to the Level 1 predictions matrix. This gives our meta models a more nuance when deciding which base model to trust.

In [ ]:
# INJECT L2 RAW FEATURES

print("\nInjecting Level 2 raw features (AIRCRAFT, SIZE, PERSON)...")
L2_RAW_FEATURES = ["AIRCRAFT", "SIZE", "PERSON"]
for col in L2_RAW_FEATURES:
    level2_input[col] = X_test[col].astype(str)

l2_te_path = RUN_DIR / "models" / "l2_target_encoder.joblib"
if l2_te_path.exists():
    l2_te_global = joblib.load(l2_te_path)
    level2_input[L2_RAW_FEATURES] = l2_te_global.transform(level2_input[L2_RAW_FEATURES])
    for col in L2_RAW_FEATURES:
        level2_input[col] = level2_input[col].astype("float64")
else:
    print("Missing Level 2 Target Encoder. Proceeding without encoding (may crash).")



Injecting Level 2 raw features (AIRCRAFT, SIZE, PERSON)...


## Level 2 Inference (Meta-Models)
Now we run our Level 2 models. These models look at the predictions made by all the Level 1 models (plus the extra injected features) and learn how to correct their mistakes. Just like before, we load every fold, run the inference, and average the probabilities.

In [26]:
# LEVEL 2 INFERENCE

print(f"\n{'='*72}")
print("  Level 2 Inference")
print(f"{'='*72}")

level3_input = {}

for l2_model_name in LEVEL2_MODEL_NAMES:
    fold_probs_sum = np.zeros(n_test, dtype=float)
    folds_loaded = 0

    for fold in range(1, N_FOLDS + 1):
        model_path = RUN_DIR / "models" / f"{l2_model_name}_fold{fold}.joblib"
        if not model_path.exists():
            print(f"  ⚠ Missing: {model_path}")
            continue

        model = joblib.load(model_path)
        fold_probs = predict_probs(model, level2_input)
        fold_probs_sum += fold_probs
        folds_loaded += 1

        del model

    if folds_loaded == 0:
        raise FileNotFoundError(f"No fold models found for {l2_model_name}")

    avg_probs = fold_probs_sum / folds_loaded
    level3_input[l2_model_name] = avg_probs

    print(
        f"  {l2_model_name}: {folds_loaded} folds loaded, "
        f"mean_prob={avg_probs.mean():.4f}, "
        f"pred_pos_rate={(avg_probs >= 0.5).mean():.4f}"
    )

gc.collect()



  Level 2 Inference
  l2_logistic_baseline: 3 folds loaded, mean_prob=0.2626, pred_pos_rate=0.1706
  l2_xgb_standard: 3 folds loaded, mean_prob=0.1224, pred_pos_rate=0.0902


100

## Level 3 Blending & Thresholding
Instead of training another complex model for Level 3, we use a simple weighted average (blending). We apply the specific weights we found during training to combine the Level 2 predictions into one final probability.

Since our dataset is imbalanced (damage is relatively rare), using the default 0.5 threshold to classify damage vs no-damage isn't optimal. We apply the custom threshold we optimized during cross-validation to convert our final probability into a hard 0 or 1 prediction.

In [27]:
# LEVEL 3: WEIGHTED AVERAGE BLENDER

print(f"\n{'='*72}")
print("  Level 3: Weighted Average Blender")
print(f"{'='*72}")

# Blend Level 2 probabilities
# apply our optimized 50/50 blending weights
l3_blended = sum(level3_input[name] * weight for name, weight in l3_weights.items())

print(f"L3 weights: {l3_weights}")
print(f"L3 threshold: {l3_threshold:.6f}")
print(f"Blended mean probability: {l3_blended.mean():.4f}")
print(f"Blended std: {l3_blended.std():.4f}")

# Apply threshold to get binary predictions
predictions = (l3_blended >= l3_threshold).astype(int) # apply the optimal threshold we found during cross validation

n_damage = int(predictions.sum())
n_no_damage = int((predictions == 0).sum())
damage_rate = n_damage / len(predictions) if len(predictions) > 0 else 0.0

print(f"\nPrediction summary:")
print(f"  Total records: {len(predictions):,}")
print(f"  Damage (1): {n_damage:,} ({damage_rate:.2%})")
print(f"  No damage (0): {n_no_damage:,} ({1 - damage_rate:.2%})")



  Level 3: Weighted Average Blender
L3 weights: {'l2_logistic_baseline': 0.5, 'l2_xgb_standard': 0.5}
L3 threshold: 0.323333
Blended mean probability: 0.1925
Blended std: 0.2115

Prediction summary:
  Total records: 34,131
  Damage (1): 6,878 (20.15%)
  No damage (0): 27,253 (79.85%)


## Generating the Submission File
Finally, we take our binary predictions, attach them to the `INDEX_NR` column, and format everything how Kaggle expects it. The result is saved as `submission.csv` right inside our run directory, ready to be uploaded!

In [28]:
# BUILD AND SAVE SUBMISSION CSV

print(f"\n{'='*72}")
print("  Building submission CSV")
print(f"{'='*72}")

submission = pd.DataFrame(
    {
        "INDEX_NR": test_index_nr.values,
        "INDICATED_DAMAGE": predictions,
    }
)

# Sort by INDEX_NR ascending
submission = submission.sort_values("INDEX_NR", ascending=True).reset_index(drop=True)

# Save
output_path = RUN_DIR / "submission.csv"
submission.to_csv(output_path, index=False)

print(f"Submission saved to: {output_path}")
print(f"  Rows: {len(submission):,}")
print(f"  Columns: {list(submission.columns)}")
print(f"  INDEX_NR range: {submission['INDEX_NR'].min()} — {submission['INDEX_NR'].max()}")
print(f"  INDEX_NR sorted ascending: {submission['INDEX_NR'].is_monotonic_increasing}")
print(f"  INDICATED_DAMAGE unique values: {sorted(submission['INDICATED_DAMAGE'].unique())}")
print(f"\nFirst 10 rows:")
print(submission.head(10).to_string(index=False))
print(f"\nLast 10 rows:")
print(submission.tail(10).to_string(index=False))
print(f"\n{'='*72}")
print("  Inference complete!")
print(f"{'='*72}")



  Building submission CSV
Submission saved to: runs/2026-04-26_16-52-21/submission.csv
  Rows: 34,131
  Columns: ['INDEX_NR', 'INDICATED_DAMAGE']
  INDEX_NR range: 9000000 — 9034130
  INDEX_NR sorted ascending: True
  INDICATED_DAMAGE unique values: [np.int64(0), np.int64(1)]

First 10 rows:
 INDEX_NR  INDICATED_DAMAGE
  9000000                 0
  9000001                 0
  9000002                 0
  9000003                 0
  9000004                 1
  9000005                 1
  9000006                 0
  9000007                 0
  9000008                 1
  9000009                 1

Last 10 rows:
 INDEX_NR  INDICATED_DAMAGE
  9034121                 0
  9034122                 0
  9034123                 0
  9034124                 0
  9034125                 0
  9034126                 0
  9034127                 0
  9034128                 0
  9034129                 0
  9034130                 0

  Inference complete!
